# Perfect secrecy

In [51]:
import random
import secrets
from string import ascii_lowercase, ascii_uppercase


Notation, parameters:
- key space: $\mathcal{K}$ finite set of possible keys, probabilistic algorithm
- message space: $\mathcal{M}$ is the set of possible messages
- ciphertext space: $\mathcal{C}$ is the set of possible encrypted messages
- we allow $\mathtt{Enc}$ to be a probabilistic algorithm
- we also expect $\mathtt{Dec}_k(\mathtt{Enc}_k(m)) = m$

Other definitions:
- Let $K$ be a random variable over $\mathcal{K}$: $Pr(K = k)$ is the probability that the $\mathtt{Gen}$ algorithm selected the $k \in \mathcal{K}$ key
- Let $M$ be a random variable over $\mathcal{M}$: $Pr(M = m)$ is the probability of choosing the message $m \in \mathcal{M}$ for encryption
- We assume that $M$ and $K$ are independent

**Task**: Let $\mathcal{K} = \{0, \dots, 25\}$ and $Pr(K = k) = 1/26$ for all $k \in \mathcal{K}$. We apply a shift cipher. Assume that the distribution of possible messages is $Pr(M = \mathtt{a}) = 0.7$ and $Pr(M = \mathtt{z}) = 0.3$.

1. What is the probability of receiving the encrypted message $\mathtt{B}$?
2. What is the probability that the message $\mathtt{a}$ was encrypted, given that $\mathtt{B}$ was the ciphertext?

 **Solution**:

 1.


 **Definition** (Perfect security): A $(\mathtt{Gen}, \mathtt{Enc}, \mathtt{Dec})$ encryption scheme with message space $\mathcal{M}$ is perfectly secure if all distributions over $M$, for any message $m \in \mathcal{M}$ and for any ciphertext $c \in \mathcal{C}$ for which $Pr(C = c) > 0$: $$Pr(M = m \,|\,C = c) = Pr(M = m)$$

 ## Alternative definition of security

 (This is equivalent to the previous definition)

 Let $\Pi = (\mathtt{Gen}, \mathtt{Enc}, \mathtt{Dec})$ be an encryption scheme with message space $\mathcal{M}$. Let $\mathcal{A}$ be an adversary. The eavesdropping indistinguishability experiment $\mathtt{PrivK}^{\text{eav}}_{\mathcal{A},\Pi}$:
 1. the attacker $\mathcal{A}$ issues $m_0,m_1$ messages, to which $|m_0| = |m_1|$ and $m_0, m_1 \in \mathcal{M}$
 2. the key $k$ is randomly generated with the $\mathtt{Gen}$ algorithm and we choose a $b \in_R \{0,1\}$ bit. Let $c = \mathtt{Enc}_k(m_b)$, which we pass to $\mathcal{A}$.
 3. the attacker $\mathcal{A}$ outputs a bit $b' \in \{0,1\}$
 4. If $b' = b$, then $\mathtt{PrivK}^{\text{eav}}_{\mathcal{A},\Pi} = 1$ and the experiment succeeds, otherwise $\mathtt{PrivK }^{\text{eav}}_{\mathcal{A},\Pi} = 0$ and the experiment fails.

 **Theorem**: An encryption scheme $\Pi$ is perfectly secure over $\mathcal{M}$ if an attacker $\forall \mathcal{A}$ $$Pr(\mathtt{PrivK}^{\text{eav }}_{\mathcal{A},\Pi} = 1) = \frac{1}{2}.$$

 **Task**: Let $\mathcal{M} = \{a,b\}^2$ and $|k| = t \in_R \{1, 2\}$. Show that Vigenère's encryption *does not* satisfy the above definition, i.e. specify an attacker for whom $Pr(\mathtt{PrivK}^{\text{eav}}_{\mathcal{A},\ Pi} = 1) > \frac{1}{2}.$

 **Solution**:


 ## One-Time Pad

 Gilbert Vernam, 1926, but existed before.

 Let $n > 0$ be fixed. Then $\mathcal{K} = \mathcal{M} = \mathcal{C} = \{0,1\}^n$.
 - $\mathtt{Gen}$: chooses a key $k \in \mathcal{K}$ based on a uniform distribution
 - $\mathtt{Enc}$: given key $k \in \{0, 1\}^n$ and message $m \in \{0,1\}^n$, $c := k \oplus m$
 - $\mathtt{Dec}$: given key $k \in \{0, 1\}^n$ and ciphertext $c \in \{0,1\}^n$, $m := k \oplus c$

 **Task**: Show that the encryption scheme above is correct!

 **Solution**:

 **Task**: Why is the one-time pad impractical?

In [52]:
def otp_gen(k: int) -> bytes:
    return random.randbytes(5)


In [53]:
def otp_enc(key: bytes, plaintext: bytes) -> bytes:
    return bytes([k ^ m for k, m in zip(key, plaintext)])


In [54]:
def otp_dec(key: bytes, ciphertext: bytes) -> bytes:
    return bytes([k ^ c for k, c in zip(key, ciphertext)])


In [55]:
key = otp_gen(5)
key


b'\xdc\x96\xde\x86]'

In [56]:
otp_dec(key, otp_enc(key, b"hello"))


b'hello'

Using English alphabet characters:

In [57]:
def repeat_to_length(s: str, length: int) -> str:
    return (s * (length // len(s) + 1))[:length]


def vigenere_gen(length: int) -> str:
    return "".join(random.choices(ascii_lowercase, k=length))


def vigenere_enc(key: str, plaintext: str) -> str:
    c = ""
    key = repeat_to_length(key, len(plaintext))
    for k, char in zip(key, plaintext):
        k_i = ascii_lowercase.find(k)
        char_i = ascii_lowercase.find(char)
        c += ascii_uppercase[(k_i + char_i) % 26]
    return c


def vigenere_dec(key: str, ciphertext: str) -> str:
    p = ""
    key = repeat_to_length(key, len(ciphertext))
    for k, char in zip(key, ciphertext):
        k_i = ascii_lowercase.find(k)
        char_i = ascii_uppercase.find(char)
        p += ascii_lowercase[(char_i - k_i) % 26]
    return p


In [58]:
key = vigenere_gen(5)
vigenere_enc(key, "hello")


'RYBWP'

Notice the following:

In [59]:
key = 'kcqyzhepxautiqekxejmoretzhztrwwqdylbttvejmedbsanybpxqik'
vigenere_enc(key, 'ifyouwanttosurviveouthereyouvegottoknowwhereyourtowelis')


'SHOMTDECQTILCHZSSIXGHYIKDFNNMACEWRZLGHRAQQVHZGUERPLBBQC'

In [60]:
key = 'zakavkxolfqdlzhwsqjbzmtwmmnakwurwexdcuywksgorghnnedvtcp'
vigenere_enc(key, 'themythofosiriswasofimportanceinancientegyptianreligion')


'SHOMTDECQTILCHZSSIXGHYIKDFNNMACEWRZLGHRAQQVHZGUERPLBBQC'

 ## Algebra of encryption schemes

 Claude Shannon (1949)

 **Definition**: Let $\mathcal{M}$ be a message space, $\mathcal{C}$ and $\mathcal{C}'$ two ciphertext spaces. Let $\mathbb{S}_1$ and $\mathbb{S}_2$ be two encryption schemes such that $$\mathtt{Enc}_{\mathbb{S}_1} : \mathcal{M} \rightarrow \mathcal {C}\quad \text{and}\quad \mathtt{Enc}_{\mathbb{S}_2} : \mathcal{C} \rightarrow \mathcal{C}'.$$ Then $\mathbb{S} = \mathbb{S}_1 \times \mathbb{S}_2$ will be a new encryption scheme and $\mathtt{Enc}_{\mathbb{S}} : \mathcal{M} \rightarrow \mathcal{C}' $. During the encryption, we first use $\mathtt{Enc}_{\mathbb{S}_1}$ and then $\mathtt{Enc}_{\mathbb{S}_2}$.

 This is always associative: $(\mathbb{S}_1 \times \mathbb{S}_2) \times \mathbb{S}_3 = \mathbb{S}_1 \times (\mathbb{S}_2 \times \mathbb{ S}_3)$

 # Tasks

 1. Let $\mathcal{K} = \{0,\dots,25\}$ and $Pr(K = k) = 1/26$ for every $k \in \mathcal{K}$ where $k$ is the encryption key for the shift cipher. Suppose that the distribution of possible messages is as follows: $$Pr(M = \mathtt{kim}) = 0.5,\; Pr(M = \mathtt{ann}) = 0.2,\;Pr(M = \mathtt{boo}) = 0.3$$
     1. What is the probability that $C = \mathtt{DQQ}$?
     2. What is the probability that the message $M = \mathtt{ann}$ was encrypted, given that $C = \mathtt{DQQ}$ was the encrypted message?
 2. Show that the shift cipher is perfectly secure if exactly one character is encrypted!
 3. Show that the one-time pad encryption scheme is perfectly secure!
 4. An encryption scheme is idempotent if $\mathbb{S}^2 = \mathbb{S} \times \mathbb{S} = \mathbb{S}$. Show that shift cipher is idempotent.
 5. Let $\mathbb{S}_1$ and $\mathbb{S}_2$ be idempotent encryption schemes. Assume that $\mathbb{S}_1 \times \mathbb{S}_2 = \mathbb{S}_2 \times \mathbb{S}_1$. Show that $\mathbb{S}_1 \times \mathbb{S}_2$ is idempotent!